In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ── Part A: LSTM Text Classification ──────────────────────────
sentences = ["I love this movie", "Terrible film", "Amazing acting",
             "Worst movie ever", "Highly recommend", "Complete waste"]
labels    = [1, 0, 1, 0, 1, 0]

vocab = {w: i+2 for i, w in enumerate(set(" ".join(sentences).split()))}
vocab["<pad>"] = 0; vocab["<unk>"] = 1

def encode(s, max_len=8):
    ids = [vocab.get(w, 1) for w in s.split()]
    return ids[:max_len] + [0] * (max_len - len(ids))

class TextDataset(Dataset):
    def __init__(self): pass
    def __len__(self): return len(sentences)
    def __getitem__(self, i):
        return torch.tensor(encode(sentences[i])), torch.tensor(labels[i])

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hid=32):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hid, batch_first=True)
        self.fc   = nn.Linear(hid, 2)
    def forward(self, x):
        out, _ = self.lstm(self.emb(x))
        return self.fc(out[:, -1])

loader = DataLoader(TextDataset(), batch_size=2)
clf    = LSTMClassifier(len(vocab))
opt    = torch.optim.Adam(clf.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(20):
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(clf(xb), yb).backward()
        opt.step()
print("Classification training done ✓")

# ── Part B: Seq2Seq ────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab, emb, hid):
        super().__init__()
        self.emb  = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hid, batch_first=True)
    def forward(self, x):
        return self.lstm(self.emb(x))

class Decoder(nn.Module):
    def __init__(self, vocab, emb, hid):
        super().__init__()
        self.emb  = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hid, batch_first=True)
        self.fc   = nn.Linear(hid, vocab)
    def forward(self, x, h, c):
        out, (h, c) = self.lstm(self.emb(x.unsqueeze(1)), (h, c))
        return self.fc(out.squeeze(1)), h, c

class Seq2Seq(nn.Module):
    def __init__(self, vocab=20, emb=16, hid=32):
        super().__init__()
        self.enc = Encoder(vocab, emb, hid)
        self.dec = Decoder(vocab, emb, hid)
    def forward(self, src, trg):
        _, (h, c) = self.enc(src)
        outputs = []
        x = trg[:, 0]
        for t in range(1, trg.shape[1]):
            out, h, c = self.dec(x, h, c)
            outputs.append(out.unsqueeze(1))
            x = trg[:, t]
        return torch.cat(outputs, dim=1)

VOCAB = 20
src = torch.randint(1, VOCAB, (4, 6))
trg = torch.randint(1, VOCAB, (4, 5))

s2s  = Seq2Seq(VOCAB)
opt2 = torch.optim.Adam(s2s.parameters())
loss_fn2 = nn.CrossEntropyLoss()

for epoch in range(50):
    opt2.zero_grad()
    out = s2s(src, trg)                        # (4, 4, VOCAB)
    loss = loss_fn2(out.reshape(-1, VOCAB), trg[:, 1:].reshape(-1))
    loss.backward(); opt2.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

Classification training done ✓
Epoch 10 | Loss: 2.9296
Epoch 20 | Loss: 2.8143
Epoch 30 | Loss: 2.6588
Epoch 40 | Loss: 2.4228
Epoch 50 | Loss: 2.1054
